In [1]:
import pandas as pd
import numpy as np


In [2]:
file_path = r"C:\Users\karen\Documents\HumanidadesDigitales_git\BDD_Corpus\Corpus_completo_revisado.xlsx"
corpus_completo = pd.read_excel(file_path, engine='openpyxl')

In [3]:
# Formatear fechas

meses = {
    'enero': '01',
    'febrero': '02',
    'marzo': '03',
    'abril': '04',
    'mayo': '05',
    'junio': '06',
    'julio': '07',
    'agosto': '08',
    'septiembre': '09',
    'octubre': '10',
    'noviembre': '11',
    'diciembre': '12'
}

def convertir_fecha(fecha_str):
    partes = fecha_str.split(' de ')
    dia = partes[0].zfill(2)  # Asegura 2 dígitos para días 1-9
    mes = meses[partes[1]]
    año = partes[2]
    return f"{dia}/{mes}/{año}"  # Formato DD/MM/AAAA

corpus_completo['Fecha'] = corpus_completo['Fecha'].apply(convertir_fecha)
corpus_completo['Fecha'] = pd.to_datetime(corpus_completo['Fecha'], format='%d/%m/%Y', errors='coerce')

In [4]:
corpus_completo

,Diario,Autor,Fecha,Título,Texto,Vínculo,ID
0,El Espectador,Gonzalo Hernández,2018-01-01,Fajardo: para nada tibio,"La Coalición Colombia –Partido Alianza Verde, ...",https://web.archive.org/web/20180102104221/htt...,1
1,El Espectador,Eduardo Barajas Sandoval,2018-01-01,Macedonia de Norte,Las interpretaciones de la historia sirven com...,https://web.archive.org/web/20180102104221/htt...,2
2,El Espectador,Daniel Emilio Rojas Castro,2018-01-01,El nacionalismo según Vargas Llosa,La semana pasada Mario Vargas Llosa publicó un...,https://web.archive.org/web/20180102104221/htt...,3
3,El Espectador,Reinaldo Spitaletta,2018-01-01,"Tiempo sagrado, tiempo profano","Pudiera decirse, sin ser una verdad absoluta, ...",https://web.archive.org/web/20180102104221/htt...,4
4,El Espectador,Aura Lucía Mera,2018-01-01,La rebelión de los bueyes,Lo mejor del encierro de Las Ventas fueron los...,https://web.archive.org/web/20180102104221/htt...,5
...,...,...,...,...,...,...,...
13671,Semana,Gonzalo Sánchez,2019-06-04,Es hora de salir de la trampa,Colombia tiene que liberarse del debate sobre ...,https://web.archive.org/web/20190605212054/htt...,13672
13672,Semana,Álvaro Jiménez,2019-06-03,La semana de los propietarios,"Hemos avanzado mucho como país, para silenciar...",https://web.archive.org/web/20190605212117/htt...,13673
13673,Semana,Jairo Gómez,2019-06-03,"Presidente, deje el estrés a un lado y gobierne","No pretendo, ni más faltaba, diseñarle las pol...",https://web.archive.org/web/20190605212129/htt...,13674
13674,Semana,Ana María Ruiz,2019-06-03,Era el mejor oficio del mundo,El periodismo que olisquea y escarba la realid...,https://web.archive.org/web/20190605220917/htt...,13675


In [9]:
# Contar número de documentos por año
conteo_años = corpus_completo['Fecha'].dt.year.value_counts().sort_index().reset_index()
conteo_años

,Fecha,count
0,2018,5151
1,2019,4018
2,2020,4507


In [11]:
# Contar número de documentos por diario
conteo_diarios = corpus_completo['Diario'].value_counts().reset_index()
conteo_diarios.columns = ['Diario', 'Número de documentos']
conteo_diarios

,Diario,Número de documentos
0,El Espectador,10509
1,El Tiempo,1982
2,Semana,1185


In [12]:
# Número de autores diferentes por diario
autores_por_diario = corpus_completo.groupby('Diario')['Autor'].nunique().reset_index()
autores_por_diario.columns = ['Diario', 'Número de autores diferentes']
autores_por_diario

,Diario,Número de autores diferentes
0,El Espectador,264
1,El Tiempo,249
2,Semana,105


In [13]:
# Numero de autores diferentes por año
autores_por_año = corpus_completo.groupby(corpus_completo['Fecha'].dt.year)['Autor'].nunique().reset_index()
autores_por_año.columns = ['Año', 'Número de autores diferentes']
autores_por_año

,Año,Número de autores diferentes
0,2018,380
1,2019,399
2,2020,319


In [16]:
# Total autores diferentes en el corpus
total_autores_diferentes = corpus_completo['Autor'].nunique()
total_autores_diferentes

594

In [19]:
# Cantidad de palabras por documento
Cantidad_palabras = corpus_completo['Texto'].apply(lambda x: len(str(x).split()))
Cantidad_palabras.sum()

8620835

In [25]:
Cantidad_palabras.mean()

630.3623135419714

In [22]:
# Cantidad de palabras diferentes en todo el corpus
palabras_diferentes = set()
for texto in corpus_completo['Texto']:
    palabras = str(texto).split()
    palabras_diferentes.update(palabras)
    
len(palabras_diferentes)

342988

# Topic Modeling Bertopic

In [28]:
%pip install bertopic tf-keras transformers -q 

Note: you may need to restart the kernel to use updated packages.


In [31]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from spacy.lang.es.stop_words import STOP_WORDS


In [30]:
sentence_model = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")

In [32]:
topic_model = BERTopic(language="spanish",
    top_n_words=10,          # Palabras por tópico
    min_topic_size=20,       # Tamaño mínimo del cluster
    nr_topics=50,        # Número de tópicos
    calculate_probabilities=True,
    vectorizer_model=TfidfVectorizer(stop_words=list(STOP_WORDS)),
    embedding_model= sentence_model 
    )

In [33]:
texto_seleccionado = corpus_completo['Texto'].astype(str).tolist()
#texto_seleccionado = columnas_procesadas

CORRER = True
if CORRER:
    topics, probs = topic_model.fit_transform(texto_seleccionado)

In [37]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7796,-1_país_colombia_gobierno_años,"[país, colombia, gobierno, años, política, pre...",[No sería inexacto decir que uno de los factor...
1,0,578,0_presidente_trump_duque_gobierno,"[presidente, trump, duque, gobierno, biden, pa...",[Hacer análisis mesurados sobre los primeros 1...
2,1,509,1_virus_coronavirus_pandemia_covid,"[virus, coronavirus, pandemia, covid, salud, 1...",[Luego de seis meses de aparición del coronavi...
3,2,454,2_vida_mundo_tola_amor,"[vida, mundo, tola, amor, pa, casa, tiempo, di...",[Desde hace 23 años hemos dedicado en el Insti...
4,3,408,3_colombia_país_política_gobierno,"[colombia, país, política, gobierno, político,...",[Puede que la voz de uno no altere nada. Me ne...
5,4,322,4_educación_universidad_estudiantes_universidades,"[educación, universidad, estudiantes, universi...",[En el año 1993 el expresidente César Gaviria ...
6,5,285,5_paz_farc_jep_guerra,"[paz, farc, jep, guerra, guerrilla, proceso, c...",[Hay actualmente en desarrollo una nueva amena...
7,6,249,6_fiscal_fiscalía_martínez_nhm,"[fiscal, fiscalía, martínez, nhm, odebrecht, d...",[Supongamos que el video en el que aparece Pet...
8,7,235,7_internet_digital_información_datos,"[internet, digital, información, datos, televi...",[Resulta difícil hacer seguimiento a la avalan...
9,8,228,8_fútbol_jugadores_gol_equipo,"[fútbol, jugadores, gol, equipo, goles, torneo...","[La economía devastada, las fuerzas de ocupaci..."


In [40]:
import plotly.express as px

# Obtener información de tópicos
topic_info = topic_model.get_topic_info()

# Filtrar outliers (tópico -1)
topic_info_filtered = topic_info[topic_info['Topic'] != -1]


# Gráfico de barras interactivo
fig = px.bar(topic_info,
             x='Topic',
             y='Count',
             color='Name',
             hover_data=['Representation'],
             title='Distribución de Tópicos',
             labels={'Count': 'N° Columnas', 'Name': 'Palabras Clave'},
             height=600)
fig.update_layout(xaxis={'type': 'category'})
fig.show()